# Socio-Economic Determinants of Healthcare Quality

This notebook reproduces the statistical tests and figures used in the GitHub Pages report.

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns

DATA_PATH = '../data/healthcare_data.csv'
df = pd.read_csv(DATA_PATH)
df.head()

In [ ]:
# ANOVA: star_rating by hospital_type
ac = df[df['hospital_type']=='Acute Care Hospitals']['star_rating'].dropna()
ca = df[df['hospital_type']=='Critical Access Hospitals']['star_rating'].dropna()
f_stat, p_val = stats.f_oneway(ac, ca)
(ac.mean(), ca.mean(), f_stat, p_val)

In [ ]:
# Chi-square: income_group x star_group
income_median = df['community_income'].median()
df['income_group'] = np.where(df['community_income']>=income_median,'High income','Low income')
df['star_group'] = np.where(df['star_rating']>=4,'4-5 stars','1-3 stars')
ct = pd.crosstab(df['income_group'], df['star_group'])
chi2, p, dof, exp = stats.chi2_contingency(ct)
ct, chi2, p

In [ ]:
# OLS: readmit_fail_count ~ community_income + community_population
reg_df = df[['readmit_fail_count','community_income','community_population']].dropna()
y = reg_df['readmit_fail_count']
X = sm.add_constant(reg_df[['community_income','community_population']])
model = sm.OLS(y,X).fit()
model.summary()

In [ ]:
# Robust SE (HC3)
rob = model.get_robustcov_results(cov_type='HC3')
rob.summary()

In [ ]:
# Correlation matrix
corr_cols = ['star_rating','readmit_success_count','readmit_fail_count','community_income','community_population']
df[corr_cols].corr()

In [ ]:
# Figures used in report
plt.figure(); sns.boxplot(data=df, x='hospital_type', y='star_rating'); plt.xticks(rotation=35, ha='right'); plt.title('Hospital Type vs. Overall Quality Rating'); plt.tight_layout(); plt.show()